# Workshop #3 — Silver Layer Data Quality Findings & Statistical Justification
**Course:** Programación para Análisis de Datos (Data Analysis Programming) — Semester 2026-II  
**Team 1:** Natural Hazard Monitoring Systems — Earthquake Activity Monitoring  
**Authors:** Juan Córdoba (20211020097), Angelo Ibáñez (20212020007), Mauricio Sánchez (20212020045)  

---

## Objective
This notebook performs a comprehensive data quality audit across all accumulated **Silver Parquet files** (`datalake_silver/`), computing descriptive statistics, null and pinned-default rates, IQR/Z-score outlier distributions, and regional text length statistics (`place`) using Pandas to justify the Data Governance KPIs implemented in the Gold layer.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SILVER_DIR = Path("datalake_silver") if Path("datalake_silver").exists() else Path("../datalake_silver")
silver_files = sorted(SILVER_DIR.glob("*.parquet"))
print(f"Found {len(silver_files)} Silver Parquet partition(s):")
frames = []
for f in silver_files:
    part = pd.read_parquet(f)
    print(f"  - {f.name}: {len(part):,d} rows ({f.stat().st_size / 1024.0:.2f} KB)")
    frames.append(part)

raw_silver_df = pd.concat(frames, ignore_index=True)
df = raw_silver_df.drop_duplicates(subset=["event_id"], keep="last").copy()
df["source_id"] = np.select(
    [
        df["event_id"].str.startswith("usgs_fdsn_"),
        df["event_id"].str.startswith("emsc_fdsn_"),
        df["event_id"].str.startswith("usgs_realtime_"),
        df["event_id"].str.startswith("gdacs_realtime_"),
    ],
    ["usgs_fdsn", "emsc_fdsn", "usgs_realtime", "gdacs_realtime"],
    default="usgs_fdsn",
)
df["log10_energy"] = 4.8 + 1.5 * df["magnitude"]
df["place_length"] = df["place"].fillna("").astype(str).str.len()
print(f"\nTotal rows across Silver snapshots: {len(raw_silver_df):,d}")
print(f"Unique deduplicated Silver events:  {len(df):,d} (Intra-lake duplicate rate: {(len(raw_silver_df)-len(df))/len(raw_silver_df)*100:.2f}%)")


Found 6 Silver Parquet partition(s):
  - earthquakes_silver_20261117_013502.parquet: 245 rows (24.79 KB)
  - earthquakes_silver_20261117_013535.parquet: 245 rows (24.79 KB)
  - earthquakes_silver_20261117_013549.parquet: 245 rows (24.79 KB)
  - earthquakes_silver_20261117_014009.parquet: 245 rows (24.79 KB)
  - earthquakes_silver_20261117_020008.parquet: 245 rows (24.82 KB)
  - earthquakes_silver_20261120_120000.parquet: 3,018 rows (175.14 KB)

Total rows across Silver snapshots: 4,243
Unique deduplicated Silver events:  3,263 (Intra-lake duplicate rate: 23.10%)


## 1. Null Rate & Default Focal Depth Pinning Audit
Evaluates explicit null percentages per column as well as implicit geophysical imputation (fixed $10.0\text{ km}$ focal depth assigned by global catalogs when depth inversion does not converge).


In [2]:
null_rates = pd.DataFrame({
    "Null Count": df.isnull().sum(),
    "Null Rate (%)": (df.isnull().sum() / len(df) * 100).round(4)
})
print("=== Explicit Column Null Rates ===")
print(null_rates)

pinned_10km = (df["depth_km"].round(3) == 10.000)
print(f"\n=== Implicit Default Depth Pinning (depth_km == 10.0 km) ===")
print(f"Total pinned at 10.0 km: {pinned_10km.sum():,d} / {len(df):,d} ({pinned_10km.mean()*100:.2f}%)")
print("\nPinned 10.0 km breakdown by source:")
print(df.groupby("source_id").apply(lambda g: pd.Series({
    "total_events": len(g),
    "pinned_10km_count": (g["depth_km"].round(3) == 10.0).sum(),
    "pinned_10km_pct": round((g["depth_km"].round(3) == 10.0).mean() * 100, 2)
})))


=== Explicit Column Null Rates ===
                Null Count  Null Rate (%)
event_id                 0            0.0
raw_time                 0            0.0
magnitude                0            0.0
mag_type                 0            0.0
place                    0            0.0
longitude                0            0.0
latitude                 0            0.0
depth_km                 0            0.0
tsunami                  0            0.0
source_file              0            0.0
event_time               0            0.0
magnitude_type           0            0.0
energy_joules            0            0.0
source_id                0            0.0
log10_energy             0            0.0
place_length             0            0.0

=== Implicit Default Depth Pinning (depth_km == 10.0 km) ===
Total pinned at 10.0 km: 974 / 3,263 (29.85%)

Pinned 10.0 km breakdown by source:
                total_events  pinned_10km_count  pinned_10km_pct
source_id                                

## 2. Descriptive Statistics & Multi-Source Breakdown
Summarizes central tendency, dispersion, and quantiles across magnitude, focal depth, coordinates, and logarithmic radiated seismic energy ($\log_{10} E$).


In [3]:
num_cols = ["magnitude", "depth_km", "latitude", "longitude", "log10_energy", "energy_joules"]
print("=== Global Descriptive Statistics (Unique Silver Corpus) ===")
print(df[num_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))

print("\n=== Summary Metrics by Seismological Agency ===")
print(df.groupby("source_id")[["magnitude", "depth_km", "place_length"]].agg(["count", "mean", "median", "min", "max"]).round(2))


=== Global Descriptive Statistics (Unique Silver Corpus) ===
       magnitude  depth_km  latitude  longitude  log10_energy  energy_joules
count   3263.000  3263.000  3263.000   3263.000      3263.000   3.263000e+03
mean       2.988    31.083    26.924    -45.832         9.282   2.179226e+12
std        1.723    71.693    29.161    122.713         2.585   1.778239e+13
min       -0.910    -5.000   -63.902   -179.997         3.435   2.722701e+03
5%         0.540     1.031   -29.516   -169.856         5.610   4.073803e+05
25%        1.520     6.370     6.564   -147.365         7.080   1.202264e+07
50%        2.490    10.000    34.972   -115.163         8.535   3.427678e+08
75%        4.700    20.000    49.258    113.448        11.850   7.079458e+11
95%        5.200   121.668    61.780    159.647        12.600   3.981072e+12
max        6.600   626.486    69.750    179.939        14.700   5.011872e+14

=== Summary Metrics by Seismological Agency ===
               magnitude                   

## 3. Statistical Outlier Analysis (IQR & Z-Score Methods)
Computes Tukey's Interquartile Range ($1.5 \times \text{IQR}$) and standard Z-score ($|Z| > 3$) outlier rates per numeric feature to justify the `outlier_rate_per_numeric_field` Governance KPI.


In [4]:
outlier_rows = []
for col in ["magnitude", "depth_km", "log10_energy", "energy_joules"]:
    s = df[col].dropna()
    q1, med, q3 = s.quantile(0.25), s.quantile(0.50), s.quantile(0.75)
    iqr = q3 - q1
    low_b, high_b = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_outliers = ((s < low_b) | (s > high_b)).sum()
    z_scores = np.abs((s - s.mean()) / (s.std() or 1.0))
    z_outliers = (z_scores > 3.0).sum()
    outlier_rows.append({
        "Field": col,
        "Q1": round(q1, 3),
        "Median": round(med, 3),
        "Q3": round(q3, 3),
        "IQR": round(iqr, 3),
        "IQR Outliers": int(iqr_outliers),
        "IQR Outlier Rate (%)": round(iqr_outliers / len(s) * 100, 2),
        "|Z|>3 Outliers": int(z_outliers),
        "Z Outlier Rate (%)": round(z_outliers / len(s) * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_rows)
print(outlier_df.to_string(index=False))


        Field           Q1       Median           Q3          IQR  IQR Outliers  IQR Outlier Rate (%)  |Z|>3 Outliers  Z Outlier Rate (%)
    magnitude        1.520 2.490000e+00 4.700000e+00 3.180000e+00             0                  0.00               0                0.00
     depth_km        6.370 1.000000e+01 2.000000e+01 1.363000e+01           544                 16.67              59                1.81
 log10_energy        7.080 8.535000e+00 1.185000e+01 4.770000e+00             0                  0.00               0                0.00
energy_joules 12022644.346 3.427678e+08 7.079458e+11 7.079338e+11           402                 12.32              25                0.77


## 4. Text Length Distribution Analysis (`place`)
Analyzes character length and formatting differences across USGS (distance-bearing strings), EMSC (Flinn-Engdahl uppercase region names), and GDACS (alert headlines).


In [5]:
text_stats = df.groupby("source_id")["place_length"].agg(
    count="count",
    mean="mean",
    std="std",
    min="min",
    q25=lambda s: s.quantile(0.25),
    median="median",
    q75=lambda s: s.quantile(0.75),
    max="max"
).round(2)
print("=== Place Description Character Length Distribution by Source ===")
print(text_stats)

print("\n=== Sample Place Strings per Source ===")
for src, grp in df.groupby("source_id"):
    sample_places = grp["place"].head(2).tolist()
    print(f"  [{src}]: {sample_places}")


=== Place Description Character Length Distribution by Source ===
                count    mean    std  min    q25  median    q75  max
source_id                                                           
emsc_fdsn         627   21.98   8.01    4   15.0    24.0   28.0   32
gdacs_realtime     12  115.17  10.82  102  104.0   115.5  126.0  131
usgs_fdsn         863   29.50   5.96    9   26.0    29.0   33.0   50
usgs_realtime    1761   27.60   4.56    9   25.0    27.0   30.0   55

=== Sample Place Strings per Source ===
  [emsc_fdsn]: ['SOUTH SANDWICH ISLANDS REGION', 'SOUTHEAST INDIAN RIDGE']
  [gdacs_realtime]: ['Green earthquake (Magnitude 5.7M, Depth:24.173km) in Japan 09/11/2026 04:52 UTC, 230 thousand in MMI IV.', 'Green earthquake (Magnitude 6.4M, Depth:17km) in Solomon Is. 09/11/2026 03:34 UTC, 8 thousand in MMI VI.']
  [usgs_fdsn]: ['38 km WNW of Anchor Point, Alaska', '30 km SSW of Skwentna, Alaska']
  [usgs_realtime]: ['5 km SSW of Fern Forest, Hawaii', '17 km SE of Pāhala, Hawai